# SQL Queries

This notebook explores database and answers practical questions about applications, internship completion, and hackathon performance.

In [11]:
import sqlite3
import pandas as pd

connection = sqlite3.connect('database/nextgen.db')
connection.execute('PRAGMA foreign_keys = ON;')

def run_query(query, params=None):
    return pd.read_sql_query(query, connection, params=params or ())

## Explore the Data

The relationships are: `applicants.applicant_id` connects applicants to `interns`, while `interns.intern_id` connects interns to `hackathon_scores`. This lets me trace a participant from application through internship and hackathon performance.

In [12]:
applicants_preview = run_query('SELECT * FROM applicants LIMIT 5;')
interns_preview = run_query('SELECT * FROM interns LIMIT 5;')
scores_preview = run_query('SELECT * FROM hackathon_scores LIMIT 5;')
display(applicants_preview)
display(interns_preview)
display(scores_preview)

,applicant_id,name,domain,university,application_date,status
0,APP0001,David Okeke,UI/UX Design,Covenant University,2025-01-20,Under Review
1,APP0002,Samuel Williams,UI/UX Design,Lagos State University,2025-01-21,Selected
2,APP0003,Chinedu Chukwu,Data Science,Covenant University,2025-02-03,Selected
3,APP0004,Yusuf Okeke,Cyber Security,"Federal University of Technology, Akure",2025-01-22,Selected
4,APP0005,Oluwatobi Osei,Web Development,Covenant University,2025-02-21,Selected


,intern_id,applicant_id,domain,start_date,completion_status
0,INT0001,APP0382,Artificial Intelligence,2025-03-11,Completed
1,INT0002,APP0217,UI/UX Design,2025-03-23,Completed
2,INT0003,APP0288,Artificial Intelligence,2025-03-13,Completed
3,INT0004,APP0270,Mobile App Development,2025-03-18,Completed
4,INT0005,APP0010,UI/UX Design,2025-03-10,Completed


,intern_id,score,domain
0,INT0001,96.5,Artificial Intelligence
1,INT0002,94.0,UI/UX Design
2,INT0003,91.5,Artificial Intelligence
3,INT0004,89.0,Mobile App Development
4,INT0005,87.5,UI/UX Design


In [13]:
table_counts = run_query('''
    SELECT 'applicants' AS table_name, COUNT(*) AS row_count FROM applicants
    UNION ALL SELECT 'interns', COUNT(*) FROM interns
    UNION ALL SELECT 'hackathon_scores', COUNT(*) FROM hackathon_scores
''')
table_counts

,table_name,row_count
0,applicants,420
1,interns,187
2,hackathon_scores,151


## Query 1: Completed Interns by Domain

This shows how many interns completed the program in each technical domain.

In [14]:
query_1 = '''
-- Answer: How many interns completed each domain's program?
SELECT
    domain,
    COUNT(*) AS completed_count
FROM interns
WHERE completion_status = 'Completed'
GROUP BY domain
ORDER BY completed_count DESC;
'''
query_1_result = run_query(query_1)
query_1_result

,domain,completed_count
0,Artificial Intelligence,31
1,UI/UX Design,29
2,Data Science,29
3,Cyber Security,24
4,Web Development,23
5,Mobile App Development,22


The result identifies which domains have the largest number of successfully completed internships.

## Query 2: Average Hackathon Score by Domain

Averages make it possible to compare hackathon outcomes across domains while keeping the scale between 50 and 100.

In [15]:
query_2 = '''
-- Answer: What is the average hackathon score per domain?
SELECT
    domain,
    ROUND(AVG(score), 2) AS avg_score
FROM hackathon_scores
GROUP BY domain
ORDER BY avg_score DESC;
'''
query_2_result = run_query(query_2)
query_2_result

,domain,avg_score
0,Mobile App Development,83.64
1,Artificial Intelligence,81.32
2,Web Development,81.19
3,Cyber Security,81.11
4,UI/UX Design,79.94
5,Data Science,79.86


The result identifies what is the average hackathon score per domain.

## Query 3: High-Performing Interns

The `JOIN` combines intern details with their scores using `intern_id`. Aliases (`i`, `h`, and `a`) make the query easier to read, and the applicant join adds the participant's name and university.

In [16]:
query_3 = '''
-- Answer: Which interns scored 85 or above?
SELECT
    i.intern_id,
    a.name,
    a.university,
    i.domain,
    h.score
FROM interns AS i
JOIN hackathon_scores AS h
    ON i.intern_id = h.intern_id
JOIN applicants AS a
    ON i.applicant_id = a.applicant_id
WHERE h.score >= 85
ORDER BY h.score DESC;
'''
query_3_result = run_query(query_3)
query_3_result

,intern_id,name,university,domain,score
0,INT0084,Oluwatobi Osei,Covenant University,Web Development,100.00
1,INT0104,Precious Udo,Babcock University,UI/UX Design,100.00
2,INT0080,Michael Eze,"University of Nigeria, Nsukka",Mobile App Development,98.89
3,INT0115,Samuel Osei,"Federal University of Technology, Akure",UI/UX Design,96.56
4,INT0001,Samuel Williams,Lagos State University,Artificial Intelligence,96.50
5,INT0028,Amina Okeke,University of Ibadan,Web Development,95.70
6,INT0022,Kelechi Okeke,Babcock University,UI/UX Design,94.93
7,INT0002,Chinedu Nwosu,Ahmadu Bello University,UI/UX Design,94.00
8,INT0143,Precious Bello,University of Ibadan,Mobile App Development,93.80
9,INT0138,Yusuf Okafor,Lagos State University,Data Science,92.76


This shows how many interns scored 85 or above.

## Query 4: Applied-to-Completed Conversion by Domain

The `LEFT JOIN` is essential because every applicant must remain in the denominator, including applicants who never became completed interns. An `INNER JOIN` would remove those applicants and overstate conversion. `COUNT(DISTINCT ...)` protects the entity counts from duplication, and `100.0` ensures decimal division instead of integer division. `NULLIF` safely prevents division by zero.

In [17]:
query_4 = '''
-- Answer: What is the conversion rate from applied to completed per domain?
SELECT
    a.domain,
    COUNT(DISTINCT a.applicant_id) AS total_applicants,
    COUNT(DISTINCT i.intern_id) AS total_completed,
    ROUND(
        100.0 * COUNT(DISTINCT i.intern_id)
        / NULLIF(COUNT(DISTINCT a.applicant_id), 0),
        2
    ) AS conversion_rate_pct
FROM applicants AS a
LEFT JOIN interns AS i
    ON a.applicant_id = i.applicant_id
    AND i.completion_status = 'Completed'
GROUP BY a.domain
ORDER BY conversion_rate_pct DESC;
'''
query_4_result = run_query(query_4)
query_4_result

,domain,total_applicants,total_completed,conversion_rate_pct
0,Artificial Intelligence,64,31,48.44
1,UI/UX Design,72,29,40.28
2,Cyber Security,60,24,40.00
3,Data Science,78,29,37.18
4,Web Development,65,23,35.38
5,Mobile App Development,81,22,27.16


This shows conversion rate from applied to completed per domain.
This conversion view is useful for program managers because it compares successful outcomes with the full applicant pool rather than only people who reached the internship stage.

## Query 5: University Hackathon Performance

This query follows the three-table relationship from university to applicant to intern to score. Universities with no scored interns are naturally excluded by the inner joins, preventing an average based on no observations.

In [18]:
query_5 = '''
-- Answer: Which universities produced the highest-scoring hackathon interns on average?
SELECT
    a.university,
    COUNT(DISTINCT h.intern_id) AS number_of_scored_interns,
    ROUND(AVG(h.score), 2) AS average_score,
    MAX(h.score) AS highest_score
FROM applicants AS a
JOIN interns AS i ON a.applicant_id = i.applicant_id
JOIN hackathon_scores AS h ON i.intern_id = h.intern_id
GROUP BY a.university
ORDER BY average_score DESC, highest_score DESC;
'''
query_5_result = run_query(query_5)
query_5_result

,university,number_of_scored_interns,average_score,highest_score
0,Ahmadu Bello University,20,82.91,94.00
1,"Federal University of Technology, Akure",17,82.61,96.56
2,Lagos State University,19,82.16,96.50
3,Covenant University,18,82.00,100.00
4,Babcock University,23,81.01,100.00
5,University of Lagos,17,80.05,90.93
6,University of Ibadan,15,79.47,95.70
7,"University of Nigeria, Nsukka",22,78.46,98.89


The highest average highlights universities whose scored participants performed strongest overall, while `highest_score` shows the best individual result.

## Bonus Query: Completion Versus Dropout by Domain

In [19]:
bonus_query = '''
-- Bonus: How many interns completed versus dropped out per domain?
SELECT
    domain,
    SUM(CASE WHEN completion_status = 'Completed' THEN 1 ELSE 0 END) AS completed_count,
    SUM(CASE WHEN completion_status = 'Dropped Out' THEN 1 ELSE 0 END) AS dropped_out_count
FROM interns
GROUP BY domain
ORDER BY completed_count DESC;
'''
bonus_result = run_query(bonus_query)
bonus_result

,domain,completed_count,dropped_out_count
0,Artificial Intelligence,31,5
1,UI/UX Design,29,7
2,Data Science,29,5
3,Cyber Security,24,3
4,Web Development,23,1
5,Mobile App Development,22,8


This shows how many interns completed versus dropped out per domain.

In [20]:
connection.close()